In [118]:
import pandas as pd
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, WebDriverException
import re
from datetime import datetime, timedelta
import time

options = webdriver.ChromeOptions()
driver_path = 'D:\\downloads\\chromedriver-win64\\chromedriver.exe'

def scrape_detailed_info(perfume_link, driver):
    detailed_info = {}
    try:
        print(f"Attempting to open {perfume_link}")
        driver.get(perfume_link)
        WebDriverWait(driver, 60).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div#main-content')))
        
        # Extract Year, Perfumers, Main Accords, Notes, Seasonal Preferences as before
        # Year Extraction
        try:
            description_element = driver.find_element(By.XPATH, '//*[@itemprop="description"]/p')
            description = description_element.text
            year_match = re.search(r'\b\d{4}\b', description)
            detailed_info['Year'] = year_match.group(0) if year_match else 'Unknown'
        except NoSuchElementException:
            detailed_info['Year'] = 'Unknown'
        
        # Main Accords, Notes, and Perfumers Extraction as before
        # Main Accords Extraction
        try:
            accord_elements = driver.find_elements(By.CSS_SELECTOR, 'div.accord-box > div.accord-bar')
            detailed_info['Main Accords'] = {elem.text: elem.get_attribute('style').split('width: ')[1].rstrip('%;') for elem in accord_elements}
        except NoSuchElementException:
            detailed_info['Main Accords'] = {}

        # Notes Extraction
        notes_xpath = {
            'Top Notes': "//h4[.='Top Notes']/following-sibling::div//div[contains(@style, 'text-align: center')]/div[2]",
            'Middle Notes': "//h4[.='Middle Notes']/following-sibling::div//div[contains(@style, 'text-align: center')]/div[2]",
            'Base Notes': "//h4[.='Base Notes']/following-sibling::div//div[contains(@style, 'text-align: center')]/div[2]"
        }
        for note_type, xpath in notes_xpath.items():
            try:
                elements = driver.find_elements(By.XPATH, xpath)
                detailed_info[note_type] = [element.text for element in elements if element.text]
            except NoSuchElementException:
                detailed_info[note_type] = []

        # Perfumers Extraction
        try:
            # More specific XPath to accurately target perfumer links within the designated perfumer section
            perfumer_elements = driver.find_elements(By.XPATH, "//div[contains(@class, 'perfumer')]//a | //div[@class='grid-x grid-padding-x grid-padding-y small-up-2 medium-up-2']//a")
            if perfumer_elements:
                detailed_info['Perfumers'] = ', '.join([elem.text for elem in perfumer_elements if elem.text.strip() != ""])
            else:
                detailed_info['Perfumers'] = 'Unlisted'  # Set to 'Unlisted' if no perfumers are found
        except NoSuchElementException:
            detailed_info['Perfumers'] = 'Unlisted'  # Also set to 'Unlisted' if the section doesn't exist


        
        # Seasonal and Time of Day Preferences Extraction
        preferences = {}
        preference_types = ['winter', 'spring', 'summer', 'fall', 'day', 'night']
        for pref_type in preference_types:
            try:
                # Adjust XPath to target the inner div directly
                pref_element = driver.find_element(By.XPATH, f"//span[contains(text(), '{pref_type}')]/following::div[@class='voting-small-chart-size'][1]/div/div")
                pref_style = pref_element.get_attribute('style')
                pref_width_match = re.search(r'width: (\d+\.?\d*%);', pref_style)
                pref_width = pref_width_match.group(1) if pref_width_match else '0%'
                preferences[pref_type] = pref_width
            except NoSuchElementException:
                preferences[pref_type] = '0%'

        detailed_info['Seasonal Preferences'] = preferences

    except TimeoutException:
        print(f"Timeout occurred while loading {perfume_link}")
    except WebDriverException as e:
        print(f"WebDriver exception occurred: {e}")

    
    return detailed_info
    pass



def update_csv_with_new_info(original_filename, new_filename, driver_path, options):
    if os.path.exists(original_filename):
        df_existing = pd.read_csv(original_filename).fillna('Unknown')
        if 'Year' in df_existing.columns:
            df_existing['Year'] = df_existing['Year'].astype(str)
        else:
            print("'Year' column not found in the existing dataframe.")
    else:
        print(f"No existing file found: {original_filename}")
        return

    if os.path.exists(new_filename):
        df_new = pd.read_csv(new_filename).fillna('Unknown')
        if 'Year' in df_new.columns:
            df_new['Year'] = df_new['Year'].astype(str)
        else:
            print("'Year' column not found in the new dataframe.")
    else:
        df_new = pd.DataFrame(columns=['Perfume Name', 'Perfume Brand', 'Year', 'Gender', 'Perfumers', 'Main Accords', 'Top Notes', 'Middle Notes', 'Base Notes', 'Seasonal Preferences'])

    driver = webdriver.Chrome(driver_path, options=options)
    scraped_count = 0
    scrape_limit = 1  # Adjust as needed

    try:
        for index, row in df_existing.iterrows():
            if scraped_count >= scrape_limit:
                break

            existing_entry = df_new[(df_new['Perfume Name'] == row['Perfume Name']) & (~df_new['Perfumers'].isna()) & (~df_new['Main Accords'].isna())]
            if not existing_entry.empty:
                continue  # Skip if the entry is complete

            attempt_start_time = datetime.now()
            detailed_info = {}
            while True:
                try:
                    detailed_info = scrape_detailed_info(row['Link'], driver)
                    if detailed_info:  # If detailed_info is not empty
                        break
                except TimeoutException:
                    print(f"Timeout occurred while loading {row['Link']}")
                except WebDriverException as e:
                    print(f"WebDriver exception occurred: {e}")
                    # Exiting the loop if WebDriver exception occurs
                    break

                if (datetime.now() - attempt_start_time) > timedelta(seconds=30):
                    print(f"Retrying due to inactivity for {row['Perfume Name']}.")
                    driver.quit()
                    driver = webdriver.Chrome(driver_path, options=options)
                    attempt_start_time = datetime.now()

            # Only proceed if detailed_info is not empty and WebDriver exception hasn't occurred
            if detailed_info:
                detailed_info['Perfume Name'] = row['Perfume Name']
                detailed_info['Perfume Brand'] = row['Perfume Brand']
                df_detailed_info = pd.DataFrame([detailed_info])
                df_new = pd.concat([df_new, df_detailed_info], ignore_index=True)
                scraped_count += 1
            
            # Introducing a delay between each request
            time.sleep(20)  # Delay for 5 seconds
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    finally:
        driver.quit()
        print("WebDriver closed.")

    df_new['Year'] = df_new['Year'].apply(lambda x: str(int(float(x))) if x.replace('.', '', 1).isdigit() else x)
    df_new.to_csv(new_filename, index=False)
    print(f"Updated data saved to {new_filename}")

# Example usage
options = webdriver.ChromeOptions()
# options.add_argument("--headless")  # Run in headless mode
update_csv_with_new_info("top_50_perfumes_basic_info.csv", "detailed_perfume_info.csv", "D:\\downloads\\chromedriver-win64\\chromedriver.exe", options)

'Year' column not found in the existing dataframe.
Attempting to open https://www.fragrantica.com/perfume/Viktor-Rolf/Spicebomb-Extreme-30499.html
WebDriver closed.
Updated data saved to detailed_perfume_info.csv


In [40]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementClickInterceptedException
import pandas as pd
import time

def initialize_driver():
    options = webdriver.ChromeOptions()
    # options.add_argument('--headless')  # Uncomment for headless mode
    driver = webdriver.Chrome('D:\\downloads\\chromedriver-win64\\chromedriver.exe', options=options)
    return driver

def click_show_more(driver):
    try:
        show_more_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Show more results')]"))
        )
        # Attempt to click normally
        try:
            show_more_button.click()
        except ElementClickInterceptedException:
            # If normal click is intercepted, use JavaScript to click
            driver.execute_script("arguments[0].click();", show_more_button)
        
        # Wait a moment for new results to load
        time.sleep(2)
    except (NoSuchElementException, TimeoutException):
        print("No more results button found or there are no more contents to load.")
    except Exception as e:
        print(f"Error clicking 'Show more results': {e}")

def scrape_basic_info(driver, filename):
    driver.get('https://www.fragrantica.com/search/')
    WebDriverWait(driver, 30).until(
        EC.visibility_of_all_elements_located((By.CSS_SELECTOR, 'div.cell.card.fr-news-box'))
    )
    
    perfumes_data = []
    previous_length = 0

    while True:
        # Update the list of perfume elements
        perfume_elements = driver.find_elements(By.CSS_SELECTOR, 'div.cell.card.fr-news-box')
        if len(perfume_elements) == previous_length:
            # No new perfumes loaded, attempt to click "Show more results"
            click_show_more(driver)
            time.sleep(2)  # Adjust timing based on page response
            perfume_elements = driver.find_elements(By.CSS_SELECTOR, 'div.cell.card.fr-news-box')
            if len(perfume_elements) == previous_length:
                # If still no new perfumes after clicking, break the loop
                break

        for element in perfume_elements[previous_length:]:
            # Extract perfume information here
            # Example:
            name = element.find_element(By.CSS_SELECTOR, 'p > a').text.strip()
            brand = element.find_element(By.CSS_SELECTOR, 'p > small').text.strip()
            link = element.find_element(By.CSS_SELECTOR, 'p > a').get_attribute('href')
            perfumes_data.append({"Perfume Name": name, "Perfume Brand": brand, "Link": link})

        previous_length = len(perfume_elements)

    # Save to CSV
    df = pd.DataFrame(perfumes_data)
    df.to_csv(filename, index=False)
    print(f"Data saved to {filename}")
    driver.quit()

if __name__ == "__main__":
    driver = initialize_driver()
    filename = 'top_50_perfumes_extended.csv'
    scrape_basic_info(driver, filename)


No more results button found or there are no more contents to load.
Data saved to top_50_perfumes_extended.csv


In [43]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import pandas as pd
import time

def initialize_driver():
    options = webdriver.ChromeOptions()
    driver = webdriver.Chrome('D:\\downloads\\chromedriver-win64\\chromedriver.exe', options=options)
    return driver

def click_show_more(driver):
    try:
        show_more_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Show more results')]"))
        )
        driver.execute_script("arguments[0].scrollIntoView(true);", show_more_button)
        driver.execute_script("arguments[0].click();", show_more_button)
        time.sleep(4)  # Adjust based on the site's response time
    except (NoSuchElementException, TimeoutException):
        print("No more results button found or there are no more contents to load.")
        return False  # Indicate that no more results could be loaded
    return True  # Indicate that more results were successfully loaded

def scrape_basic_info(driver, filename):
    existing_perfumes = read_existing_data(filename)
    perfumes_data = []
    driver.get('https://www.fragrantica.com/search/')
    WebDriverWait(driver, 30).until(EC.visibility_of_all_elements_located((By.CSS_SELECTOR, 'div.cell.card.fr-news-box')))
    
    while len(perfumes_data) < 50:  # Only add up to 50 new entries
        containers = driver.find_elements(By.CSS_SELECTOR, 'div.cell.card.fr-news-box')
        for container in containers:
            perfume_name_element = container.find_element(By.CSS_SELECTOR, 'p > a')
            perfume_name = perfume_name_element.text.strip()
            
            if perfume_name not in existing_perfumes:
                perfume_brand_element = container.find_element(By.CSS_SELECTOR, 'p > small')
                perfume_brand = perfume_brand_element.text.strip()
                link = perfume_name_element.get_attribute('href')
                perfumes_data.append({
                    "Perfume Name": perfume_name,
                    "Perfume Brand": perfume_brand,
                    "Link": link
                })
                existing_perfumes.append(perfume_name)
                
                if len(perfumes_data) >= 50:  # Break the loop if 50 new entries have been added
                    break

        if not click_show_more(driver):  # Stop if no more results can be loaded
            break
        
    driver.quit()
    return perfumes_data

def read_existing_data(filename):
    try:
        df = pd.read_csv(filename)
        return df['Perfume Name'].tolist()
    except FileNotFoundError:
        return []

def save_to_csv(data, filename):
    df_new = pd.DataFrame(data)
    if not df_new.empty:
        try:
            df_existing = pd.read_csv(filename)
            df_combined = pd.concat([df_existing, df_new], ignore_index=True)
        except FileNotFoundError:
            df_combined = df_new
        df_combined.to_csv(filename, index=False)
        print(f"Data updated in {filename}")
    else:
        print("No new perfumes were added.")

if __name__ == "__main__":
    driver = initialize_driver()
    filename = 'top_50_perfumes_basic_info.csv'
    new_perfumes_data = scrape_basic_info(driver, filename)
    save_to_csv(new_perfumes_data, filename)


Data updated in top_50_perfumes_basic_info.csv
